In [ ]:
# %%
# polars script with datatype toggles
# read_csv
import polars as pl
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
import os
import re
import logging
from tqdm import tqdm
 

# %%
# variables
output_fname = "processed_tab_eskd.csv"
icd_file = "/opt/data/commonfilesharePHI/ldiao/ckd_project/icd_mapping.csv"
subset = False # <<

if subset: 
    subset_size = "10000"  # 10, 100, full # <<
    output_dir = f"/opt/data/workingdir/ldiao/ckd_project/tabular_subset_{subset_size}"
    # output_dir = f"/opt/data/commonfilesharePHI/ldiao/ckd_project/ckd_tab_subset_{subset_size}"
    event_file =  f"/opt/data/workingdir/ldiao/ckd_project/tabular_subset_{subset_size}/unprocessed_tab_subset_{subset_size}.csv"
if not subset:
    # output_dir = "/opt/data/commonfilesharePHI/ldiao/ckd_project/ckd_tab_full"
    output_dir = f"/opt/data/workingdir/ldiao/ckd_project/tabular_full"
    # event_file = "/opt/data/commonfilesharePHI/jnchiang/projects/OptumCKD/CKD-Pull_v2.rpt"
    event_file = "/opt/data/commonfilesharePHI/jnchiang/projects/OptumCKD/CKD-Pull_v3.rpt.parquet"



In [ ]:

# %%
event_file

# %%

# --- data type toggles ---
use_float64 = False # True to use Float64, False for other type
use_int64 = False # True to use Int64, False for other type
# --- data types based on toggles ---
data_numeric_dtype = pl.Float64 if use_float64 else pl.Float32
data_integer_dtype = pl.Int64 if use_int64 else pl.Int16
# --- add type suffixes to output directory ---
# output_dir  += f"_{'f64' if use_float64 else 'f32'}"
# output_dir += f"_{'i64' if use_int64 else 'i16'}"

# scan vs read csv
# output_dir  += "_read" # <<

# filter ckd stage
filter_ckd_stage = True # <<
# if filter_ckd_stage: 
#     output_dir  += "_stage_filter" 



In [ ]:
try:
    os.makedirs(output_dir, exist_ok=True)
    print(f"Created output directory: {output_dir}")
except FileExistsError:
    print(f"Output directory already exists: {output_dir}")

print(f"Processing started. Output directory: {output_dir}")
print(f"Using DataNumeric data type: {'Float64' if use_float64 else 'Float32'}")
print(f"Using DataInteger data type: {'Int64' if use_int64 else 'Int16'}")


# %%
print(output_dir)

# %%
# Setup logging
log_file_path = os.path.join(output_dir, "tab_gen_m.log")
logging.basicConfig(
    filename=log_file_path,
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

logger.info(f"Processing started. Output directory: {output_dir}")



In [ ]:

# %%
# -----------------------------
# Load and preprocess with Polars
# -----------------------------
# Using pl.read_csv to load the entire file into a DataFrame
# Add a toggle to switch between separators
print(event_file)
if subset: 
    df = pl.read_csv(
        event_file,
        separator='$',
        infer_schema_length=None,
        null_values="null",
    ).unique()

if not subset: 
    csv = event_file 
    df = pl.read_parquet(csv)
    print(len(df['PatientID'].unique()))

logger.info(f"Initial DataFrame schema: {df.schema}")

df = df.with_columns(
    pl.col("PatientID").cast(pl.Utf8, strict=False),
    pl.col("EventTimeStamp").cast(pl.Utf8, strict=False),
    pl.col("DataCategory").cast(pl.Utf8, strict=False),
    pl.col("DataNumeric").cast(data_numeric_dtype, strict=False),
    pl.col("DataType").cast(pl.Utf8, strict=False),
)
print(df.shape)



In [ ]:
# -----------------------------
# ICD code -> long title mapping (ported from embedding_gen_pl_v3.py)
# -----------------------------
icd_map_df = pl.read_csv(icd_file)
icd_map_df = icd_map_df.with_columns(
    pl.col("icd_code").cast(pl.Utf8).str.replace(".", "", literal=True)
)
icd_map = dict(zip(icd_map_df["icd_code"], icd_map_df["long_title"]))
logger.info(f"Loaded {len(icd_map)} ICD code -> long_title mappings from {icd_file}")

# Clean and prepare initial columns
df = df.with_columns(
    pl.col("EventTimeStamp").str.to_datetime("%Y-%m-%d %H:%M:%S%.f", strict=False).alias("EventTimeStamp"),
    pl.col("DataCategory").fill_null("None"),
).with_columns(
    pl.col("EventTimeStamp").dt.date().alias("EventDate")
)




In [ ]:
# %%
df.shape

# %%
# -----------------------------
# Base: full patient-day index
# -----------------------------
all_days = df.select(["PatientID", "EventDate"]).unique().sort(["PatientID", "EventDate"])
all_days = all_days.drop_nulls("EventDate")

# -----------------------------
# Extract and forward-fill ICD
# -----------------------------

# -----------------------------
# CKD staging (ported from embedding_gen_pl_v3.py)
# -----------------------------
# Same mapping used in embedding_gen_pl_v3.py:
#   N18.1 -> 1, N18.2 -> 2, N18.3 -> 3, N18.4 -> 4, N18.5 -> 5,
#   N18.6 -> 6 (ESRD), N18.9 -> 0 (CKD, unspecified stage)
custom_map = {
    1: 1,
    2: 2,
    3: 3,
    4: 4,
    5: 5,
    6: 6,  # ESRD
    9: 0,  # CKD, unspecified stage
}

icd_df = (
    df.filter(
        (pl.col("DataCategory").str.contains("N18")) &
        (pl.col("DataNumeric").is_not_null())
    )
    .with_columns(
        pl.col("DataCategory")
            .str.extract(r"N18\.([1-9])", 1)
            .cast(pl.Int64)
            .replace(custom_map, default=None)
            .alias("CKD_stage_numeric")
    )
    .select("PatientID", "EventDate", "DataCategory", "CKD_stage_numeric")
)
# only keep rows where the N18 code actually resolved to a stage
icd_df = icd_df.filter(pl.col("CKD_stage_numeric").is_not_null())

icd_daywise = (
    icd_df.sort("PatientID", "EventDate")
    .group_by("PatientID", "EventDate")
    .agg([
        pl.col("DataCategory").first().alias("ICD_combined"),
        pl.col("CKD_stage_numeric").max().alias("CKD_stage_numeric"),
    ])
)
# attach the ICD long_title description, ported from embedding_gen_pl_v3.py's icd_map
icd_daywise = icd_daywise.with_columns(
    pl.col("ICD_combined")
        .str.replace(".", "", literal=True)
        .replace(icd_map, default="Unknown")
        .alias("ICD_combined_desc")
)

base_df = all_days.join(icd_daywise, on=["PatientID", "EventDate"], how="left").sort(["PatientID", "EventDate"])
# base_df.head()

# Forward-fill ICD code, description, and numeric stage
base_df = base_df.with_columns([
    pl.col("ICD_combined").forward_fill().over("PatientID"),
    pl.col("ICD_combined_desc").forward_fill().over("PatientID"),
    pl.col("CKD_stage_numeric").forward_fill().over("PatientID"),
])

# Enforce monotonic CKD staging using a cumulative maximum (same idea as
# embedding_gen_pl_v3.py's per-patient "max_stage")
base_df = base_df.with_columns(
    pl.col("CKD_stage_numeric").fill_null(0).cum_max().over("PatientID").alias("CKD_stage_numeric")
).with_columns(
    pl.col("CKD_stage_numeric").alias("CKD_stage")
)


# %%
base_df

# %%

# -----------------------------
# filter ckd stage
# -----------------------------
# CKD_stage_numeric is already clean (built from the regex + custom_map
# extraction above, forward-filled and made monotonic per patient), so
# there's no need to re-parse it from strings. We just mirror
# embedding_gen_pl_v3.py's "max_stage >= 3" per-patient filter.
def process_ckd_stage(df: pl.DataFrame, ckd_numeric_column: str = "CKD_stage_numeric",
                       patient_id_col: str = 'PatientID', filtering_stage=filter_ckd_stage):
    if ckd_numeric_column not in df.columns:
        logger.error(f"'{ckd_numeric_column}' column not found in tabular data. Cannot proceed with label generation.")
        return None

    # Remove patient-days with no stage info at all (never had a resolvable N18 code).
    base_df = df.drop_nulls(subset=[ckd_numeric_column])

    if filtering_stage:
        initial_patients = base_df.select(patient_id_col).n_unique()

        patients_to_keep = (
            base_df.group_by(patient_id_col)
            .agg(pl.col(ckd_numeric_column).max().alias("max_stage"))
            .filter(pl.col("max_stage") >= 3)
            .select(patient_id_col)
        )

        # Use an inner join to keep only the rows for the filtered patients.
        base_df = base_df.join(patients_to_keep, on=patient_id_col, how="inner")

        kept_patients = base_df.select(patient_id_col).n_unique()
        logger.info(f"Identified {kept_patients} patients with at least one visit at or above CKD stage 3.")
        logger.info(f"Filtered out {initial_patients - kept_patients} patients who never reached CKD stage 3.")
        logger.info(f"Shape of base_df after filtering for patients at or above stage 3: {base_df.shape}")

    return base_df

base_df = process_ckd_stage(base_df)
base_df.head()

# %%
# -----------------------------
# One-hot encode diagnoses (truncated ICD codes)
# v1
# -----------------------------
def truncate_icd(code):
    code = str(code).strip().replace(" ", "")
    if '.' in code:
        prefix, suffix = code.split('.', 1)
        return f"{prefix}.{suffix[0]}" if suffix else prefix
    return code

aki_icd_codes = ["N17.0", "N17.1", "N17.2", "N17.8", "N17.9"]

# diag_df = df.filter((pl.col("DataType") == "Diagnosis") & 
#     (                pl.col("DataCategory").is_in(aki_icd_codes))
# ).with_columns(
#     pl.col("DataCategory").map_elements(truncate_icd, return_dtype=pl.Utf8).alias("ICD_clean")
# ).group_by("PatientID", "EventDate").agg(pl.col("ICD_clean").unique().sort().alias("ICD_list"))

# mlb_diag = MultiLabelBinarizer()
# diag_features = mlb_diag.fit_transform(diag_df["ICD_list"])
# diag_onehot = pl.DataFrame(diag_features, schema=[f"diag_{c}" for c in mlb_diag.classes_]).cast(data_integer_dtype)
# diag_df_onehot = pl.concat([diag_df.select("PatientID", "EventDate"), diag_onehot], how="horizontal")

# base_df = base_df.join(diag_df_onehot, on=["PatientID", "EventDate"], how="left")



# %%
aki_events = df.filter(
    (pl.col("DataType") == "Diagnosis") & 
    (pl.col("DataCategory").is_in(aki_icd_codes))
).with_columns(
    pl.col("DataCategory")
        .str.replace(".", "", literal=True)
        .replace(icd_map, default="Unknown")
        .alias("AKI_ICD_Desc")
)

# 3. Sum (count) the occurrences into one column per Patient/Date, and carry
# along the human-readable ICD long_title(s) via icd_map (ported from
# embedding_gen_pl_v3.py)
aki_count = aki_events.group_by(["PatientID", "EventDate"]).agg([
    pl.len().alias("AKI_ICD_Total"),
    pl.col("AKI_ICD_Desc").unique().sort().str.concat("; ").alias("AKI_ICD_Desc"),
])

# 4. Join this single column back to your base_df
base_df = base_df.join(aki_count, on=["PatientID", "EventDate"], how="left").with_columns(
    pl.col("AKI_ICD_Total").fill_null(0) # Ensure days with no AKI are 0, not null
)

# %%
base_df.head()

# %%
# -----------------------------
# One-hot encode diagnoses (truncated ICD codes)
# v2
# -----------------------------
#  Identify AKI events ---
# aki_events = df.filter(
#     pl.col("DataCategory").str.contains("(?i)AKI|ACUTE KIDNEY INJURY")
# ).group_by("PatientID", "EventDate").agg(
#     pl.len().alias("AKI_count")
# )

# base_df = base_df.join(aki_events, on=["PatientID", "EventDate"], how="left").with_columns(
#     pl.col("AKI_count").fill_null(0)
# )

# %%
# aki_events


In [ ]:

# %%
# top lab features from the paper ---
top_lab_features = [
    "CREATININE", "GFR", "GFREST", "ALBUMIN/CREATININE RATIO", 
    "PROTEIN/CREATININE RATIO", "BUN", "PTH"
]

lab_df = df.filter(
    (pl.col("DataType") == "Labs") & 
    (pl.col("DataNumeric").is_not_null()) &
    (pl.col("DataCategory").cast(pl.Utf8).str.to_uppercase().str.contains("|".join(top_lab_features)))
).with_columns(
    pl.col("DataCategory").cast(pl.Utf8).str.to_uppercase().alias("LabCategory")
).group_by("PatientID", "EventDate", "LabCategory").agg(pl.col("DataNumeric").first())

# %%
lab_df

# %%
lab_pivot = lab_df.pivot(
    index=["PatientID", "EventDate"],
    on="LabCategory",
    values="DataNumeric",
    aggregate_function="first",
)

# %%
lab_pivot 

# %%
# Dynamically generate a dictionary for renaming the pivoted columns
rename_dict = {c: f"lab_{c}" for c in lab_pivot.columns[2:]}
lab_pivot = lab_pivot.rename(rename_dict)

base_df = base_df.join(lab_pivot, on=["PatientID", "EventDate"], how="left")



# %%
base_df 

# %%
# exclude demographics - not mentioned in eskd paper



In [ ]:
# %%
print(base_df.columns)

# %%
base_df.head()

# %%

# -----------------------------
# Final report
# -----------------------------
logger.info(f"[INFO] Final tabular shape: {base_df.shape}")
logger.info(f"[INFO] Sample features:\n{base_df.head()}")
logger.info(f"[INFO] CKD stage counts:\n{base_df['CKD_stage'].value_counts(sort=True)}")
base_df_path = os.path.join(output_dir, output_fname)
logger.info(f"Writing final DataFrame of shape {base_df.shape} to {base_df_path}")
base_df.write_csv(base_df_path)

logger.info("End of Tabular Generation")

# check csv
# Construct the full file path
final_file_path = os.path.join(output_dir, output_fname)

# Read the processed CSV file
try:
    final_df = pl.read_csv(final_file_path)
    print("File read successfully.")
    print(final_df.head())
except Exception as e:
    print(f"An error occurred while reading the file: {e}")

# %%

In [ ]:
import polars as pl
tab_path = f"./tabular_full/processed_tab_eskd.csv"    
df = pl.read_csv(tab_path)

# Overall row/patient counts
print(f"Rows: {df.shape[0]}, Patients: {df['PatientID'].n_unique()}")

# Prevalence of CKD_stage (row-level and patient-level, since patients can span stages)
print(df["CKD_stage"].value_counts(sort=True).with_columns(
    (pl.col("count") / df.shape[0] * 100).round(2).alias("pct_rows")
))

# Patient-level prevalence: each patient's max (worst) CKD_stage
patient_stage = df.group_by("PatientID").agg(pl.col("CKD_stage").max())
print(patient_stage["CKD_stage"].value_counts(sort=True).with_columns(
    (pl.col("count") / patient_stage.shape[0] * 100).round(2).alias("pct_patients")
))
